## Environment Setup

**Two conda environments are used in this notebook:**

- **`compass-hairpin`** — the default/primary environment. Everything in this notebook runs here *except* the two cells explicitly marked below.
- **`deeplabcut`** — only needed for the "DeepLabCut Import" cell and the "Analyze the Videos with DeepLabCut" cell, i.e. only when DLC tracking itself still needs to be run for one or more sessions. This project's `deeplabcut` env has DLC 2.3.11 + TensorFlow 2.10.1 on Python 3.10.18 (matched to whatever model/config you're analyzing with — verify against your own trained model's requirements if reusing this notebook for a different one).

DeepLabCut is a markerless pose estimation toolkit that uses deep learning to track user-defined body parts (Mathis et al., 2018).

### Installation Instructions

For detailed installation instructions, see the [DeepLabCut installation guide](https://deeplabcut.github.io/DeepLabCut/docs/installation.html).

---

### Reference

**Mathis, A., Mamidanna, P., Cury, K.M. et al.** (2018). DeepLabCut: markerless pose estimation of user-defined body parts with deep learning. *Nature Neuroscience*, 21, 1281–1289. https://doi.org/10.1038/s41593-018-0209-y

# Install Necessary Packages (Only need to do once)

In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

# Imports

In [1]:
# Importing the toolbox (takes several seconds)
import pandas as pd
import numpy as np
from pathlib import Path
from pylab import *
import os
import shutil
import geopandas as gpd
import scipy.stats as sp
from datetime import datetime
import sys

from pathlib import Path

# Specify path to dlc_utils file
custom_utils_path = Path(os.getcwd()).parent/'src'/'compass_labyrinth'/'behavior'/'pose_estimation'
sys.path.append(str(custom_utils_path))

import dlc_utils

# Imports for DLC

**Needed now** — 5 sessions still don't have DLC output, so this cell and the "Analyze the Videos with DeepLabCut" section further down need to actually run for those. This requires switching the notebook's kernel to the `deeplabcut` conda env for this cell and the analysis cell specifically — everything else in this notebook runs fine in `compass-hairpin`. Fill in `dlc_config_path` above with the real path to the trained model's `config.yaml` first.

In [3]:
# DeepLabCut Import
import deeplabcut
import tensorflow as tf
import keras
os.environ['DLClight'] = 'True'
print(tf.__version__)

2.10.1


### Specify Paths and Other Information

In [2]:
##### CONFIGURE PATHS FOR YOUR LOCAL SYSTEM #####

# === VIDEO FILE LOCATIONS ===
# Original combined videos (4 mice per video, cropped per-mouse downstream).
video_path_1 = r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\Media Files'
video_path_2 = ''

# Project folder: renamed videos + reorganized DLC output + everything this
# notebook produces all live here, using the SessionXXXX convention (matching
# what DLC would have produced had it run on the renamed videos directly).
source_data_path = r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis'
videofile_path = os.path.join(source_data_path, 'videos_original')

# === DEEPLABCUT CONFIGURATION ===
# Needed only for the "Analyze the Videos with DeepLabCut" cell, to run DLC
# on the 5 sessions that haven't been analyzed yet. Requires the `deeplabcut`
# conda env/kernel for that cell specifically -- everything else in this
# notebook runs fine in `compass-hairpin`.
dlc_config_path = Path(r'D:\Gladstone Dropbox\Palop Lab\Nina\DeepLabCut Projects\tensorflow_dlc2.3\Overnight Hairpins-Nina-2025-11-04\config.yaml')
dlc_scorer = 'DLC_resnet50_Overnight HairpinsNov4shuffle1_1030000'

# === OUTPUT PATHS ===
frames_path = os.path.join(source_data_path, 'frames')  # only needed for click-based boundary selection
grid_boundaries_path = os.path.join(source_data_path, 'grid_boundaries')
dlc_cropping_bounds_path = os.path.join(source_data_path, 'dlc_cropping_bounds')
dlc_results_path = os.path.join(source_data_path, 'dlc_results')
grid_path = os.path.join(source_data_path, 'grid_files')
figure_path = os.path.join(source_data_path, 'figures')

# === METADATA ===
user_metadata_file_path = r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\OvernightHairpin_DSI_E3E4NLFMice_cleaned.xlsx'
trial_type = 'Overnight Learning Trial'

# === SUMMARY ===
print("=== Path Configuration ===")
print(f"Video Source 1: {video_path_1}")
print(f"Video Source 2: {video_path_2}")
print(f"Central Video Location: {videofile_path}")
print(f"Metadata: {user_metadata_file_path}")
print(f"Figures Output: {figure_path}")

=== Path Configuration ===
Video Source 1: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\Media Files
Video Source 2: 
Central Video Location: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\videos_original
Metadata: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\OvernightHairpin_DSI_E3E4NLFMice_cleaned.xlsx
Figures Output: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\figures


# Import Mouse Metadata

In [3]:
# Add Metadata to metadata directory
mouseinfo = dlc_utils.import_cohort_metadata(user_metadata_file_path, trial_type)

# Validate the metadata
dlc_utils.validate_metadata(mouseinfo)

# Display summary of the metadata
dlc_utils.display_metadata_summary(mouseinfo)

mouseinfo.head()

Initial rows loaded: 24
Excluded 3 trials marked for exclusion
Final dataset: 21 trials

METADATA SUMMARY
Total trials: 21
Columns: ['session', 'mouseid', 'trialfolder', 'hasdlcoutput', 'exclude trial', 'run date', 'time of day', 'rack location', 'computer', 'noldus chamber', 'arena', 'noldus trial', 'sex', 'line', 'genotype', 'birth date', 'age (months)', 'hardware readout', 'x1', 'x2', 'y1', 'y2', 'notes']
Session range: 1 - 22
Missing data:
  time of day: 15 missing values
  notes: 18 missing values


,session,mouseid,trialfolder,hasdlcoutput,exclude trial,run date,time of day,rack location,computer,noldus chamber,...,line,genotype,birth date,age (months),hardware readout,x1,x2,y1,y2,notes
0,1,1230.0,Mouse_1230_Trial_1,True,no,2026-03-24T00:00:00,5:39PM,BL,1,TL,...,E3E4KI2,E4,2024-08-07T00:00:00,23.0,25.0,190,1097,60,920,"Sick, running asap"
1,2,1265.0,Mouse_1265_Trial_1,True,no,NaT,NaN,BR,1,TR,...,E3E4KI2,E4,2024-08-19T00:00:00,23.0,52.0,1494,2395,100,966,NaN
2,3,1238.0,Mouse_1238_Trial_1,True,no,NaT,NaN,TL,1,BL,...,E3E4KI2,E3NLF,2024-08-12T00:00:00,23.0,80.0,227,1116,1129,2005,NaN
3,4,1088.0,Mouse_1088_Trial_1,False,no,NaT,NaN,TR,1,BR,...,E3E4KI2,E3,2024-06-04T00:00:00,25.0,28.0,1347,2261,1113,2005,NaN
4,5,1072.0,Mouse_1072_Trial_2,True,no,2026-03-25T00:00:00,17:18:00,BL,1,TL,...,E3E4KI2,E3,2024-06-02T00:00:00,25.0,47.0,190,1097,60,920,NaN


# OPTIONAL: Copy and rename videos from original location to VIDEOFILE_PATH


**Run this** (not skipped anymore) — copies each session's video from `Media Files\` and renames it to `SessionXXXX.mp4` in `videos_original\`, so it lines up with the DLC output naming below. The default `source_naming="trial"` already matches this data's `"Trial     N.mp4"` filenames (verified against the actual `copy_and_rename_videos()` code), so this cell works unmodified.

In [6]:
now = datetime.now(); print(now)

# 1. Save first frames
copy_results = dlc_utils.copy_and_rename_videos(
    mouseinfo_df=mouseinfo,
    video_paths=[video_path_1],
    destination_path=videofile_path,
    source_naming="trial",   # "trial" | "session-dash" | "session" | "format"
    source_format=None  # used when source_naming="format", e.g. "Session-{}.mp4"
)

2026-07-24 17:07:40.526437
------------------------
Processing session=1 -> Session0001.mp4
Session0001.mp4 already exists!
------------------------
Processing session=2 -> Session0002.mp4
Session0002.mp4 already exists!
------------------------
Processing session=3 -> Session0003.mp4
Session0003.mp4 already exists!
------------------------
Processing session=4 -> Session0004.mp4
Session0004.mp4 already exists!
------------------------
Processing session=5 -> Session0005.mp4
Session0005.mp4 already exists!
------------------------
Processing session=6 -> Session0006.mp4
Session0006.mp4 already exists!
------------------------
Processing session=7 -> Session0007.mp4
Session0007.mp4 already exists!
------------------------
Processing session=8 -> Session0008.mp4
Session0008.mp4 already exists!
------------------------
Processing session=9 -> Session0009.mp4
Session0009.mp4 already exists!
------------------------
Processing session=10 -> Session0010.mp4
Session0010.mp4 already exists!
--

# Get DeepLabCut Cropping Bounds

In [7]:
# Save first frames for all videos (needed so there's something to click on)
now = datetime.now(); print(now)

print("\nSaving first frames for all videos...")
frame_results = dlc_utils.batch_save_first_frames(
    mouseinfo_df=mouseinfo,
    video_directory=videofile_path,
    frames_directory=frames_path
)

2026-07-24 17:07:46.729614

Saving first frames for all videos...


In [8]:
# Get the DLC cropping bounds for all videos (interactive -- click 4 corners per session)
now = datetime.now(); print(now)

coordinates_dict = dlc_utils.batch_get_boundary_and_cropping(
    mouseinfo_df=mouseinfo,
    frames_directory=frames_path,
    cropping_directory=dlc_cropping_bounds_path,
    boundaries_directory=grid_boundaries_path,
    reprocess_existing=False  # set to True if you'd like to redo boundaries
)

2026-07-24 17:07:52.871624
Getting boundary points and cropping coordinates for 21 sessions...
Press 'c' to skip a session, or ESC to stop completely.
✓ Session0001 already has boundary and cropping data - skipping
✓ Session0002 already has boundary and cropping data - skipping
✓ Session0003 already has boundary and cropping data - skipping

Processing Session0004 (4/21)
Chamber: BR
Status: Missing both boundary and cropping

Labyrinth Boundary Selection for Session0004
Chamber: BR
Image size: 2560 x 2048 (W x H)

Instructions:
1. Click on the 4 corners in this order:
   - Top-Left corner
   - Bottom-Left corner
   - Bottom-Right corner
   - Top-Right corner
2. Right-click to see pixel RGB values (optional)
3. Press 'q' to confirm selection after clicking 4 corners
4. Press 'r' to reset and select again
5. Press 'c' to cancel
Top-Left corner: (1346, 1106)
Click Bottom-Left corner next...
Bottom-Left corner: (1342, 2033)
Click Bottom-Right corner next...
Bottom-Right corner: (2261, 2020

In [ ]:
# Alternative: get cropping bounds for a single session (e.g. to redo just one)
coordinates_dict = dlc_utils.get_labyrinth_boundary_and_cropping(
    frames_directory=frames_path,
    cropping_directory=dlc_cropping_bounds_path,
    boundaries_directory=grid_boundaries_path,
    session='Session0001',  # change to the session you want to (re)select
    chamber_info=None
)

# Analyze the Videos with DeepLabCut

In [4]:

# Prepare sessions -- automatically skips any session that already has DLC
# output (the 16 already-copied sessions), only queues the ones missing it
sessions_to_analyze, prep_summary = dlc_utils.prepare_dlc_analysis(
    mouseinfo,
    videofile_path,
    dlc_cropping_bounds_path,
    dlc_results_path,
)

# Print preparation summary
print(f"\nPreparation Summary:")
print(f"Ready for analysis: {prep_summary['ready_for_analysis']}")
print(f"Already analyzed: {prep_summary['skipped_existing']}")
print(f"Missing videos: {prep_summary['missing_video']}")
print(f"Missing coordinates: {prep_summary['missing_coordinates']}")

# Now run DLC analysis in the correct environment
analysis_results = []
for session in sessions_to_analyze:
    print(f"\nAnalyzing {session['session_name']}...")

    deeplabcut.analyze_videos(
        dlc_config_path,
        [session['video_path']],
        shuffle=1,
        videotype=".mp4",
        save_as_csv=True,
        cropping=session['cropping_coords'],
        destfolder=session['results_path'],
    )


Analysis already exists for Session0001, skipping...
Analysis already exists for Session0002, skipping...
Analysis already exists for Session0003, skipping...
Analysis already exists for Session0004, skipping...
Analysis already exists for Session0005, skipping...
Analysis already exists for Session0006, skipping...
Analysis already exists for Session0007, skipping...
Analysis already exists for Session0008, skipping...
Analysis already exists for Session0009, skipping...
Analysis already exists for Session0010, skipping...
Analysis already exists for Session0011, skipping...
Analysis already exists for Session0012, skipping...
Analysis already exists for Session0013, skipping...
Analysis already exists for Session0014, skipping...
Analysis already exists for Session0015, skipping...
Analysis already exists for Session0016, skipping...
Analysis already exists for Session0017, skipping...
Analysis already exists for Session0018, skipping...
Analysis already exists for Session0019, skipp

# Create Grids and save as Grid Files

If DLC tracking was already done separately and crop bounds are already known for each session (as with this dataset), you can skip the "Get DeepLabCut Cropping Bounds" / "Analyze the Videos with DeepLabCut" sections above entirely and use the cell below instead.

(Note: if DLC output lives somewhere other than this project's own `dlc_results\` folder — e.g. a separate external folder structure like this dataset's `DLC_Output\Mouse_X_Trial_Y\` — that needs a one-time reorganize/copy step into `dlc_results\` first, done once outside this notebook rather than as a permanent cell here, since that folder structure is specific to how that particular DLC run was organized.)

In [ ]:
# Synthesize the same boundary-points / cropping-bounds files that
# get_labyrinth_boundary_and_cropping() would have saved from clicking --
# built directly from the X1/X2/Y1/Y2 already known for this dataset.
#
# Note on provenance: get_labyrinth_boundary_and_cropping() computes
# cropping_coords FROM clicked boundary_points, so it correctly saves
# "derived_from_boundary": True. Here the direction is reversed -- we
# already have X1/X2/Y1/Y2 and construct a boundary_points rectangle from
# them, with no human visual verification that the maze actually falls
# within those bounds in the cropped video. Recording that accurately
# (rather than copying the flag value from the real function) matters if
# anyone audits grid alignment later and wants to know which sessions were
# manually verified vs. reconstructed from metadata.
now = datetime.now(); print(now)

print(f"Synthesizing boundary/cropping files for {len(mouseinfo)} sessions...")

for _, row in mouseinfo.iterrows():
    session_name = f"Session{int(row['session']):04d}"
    X1, X2, Y1, Y2 = int(row["x1"]), int(row["x2"]), int(row["y1"]), int(row["y2"])

    boundary_points = np.array([[X1, Y1], [X1, Y2], [X2, Y2], [X2, Y1]])  # TL, BL, BR, TR
    np.save(os.path.join(grid_boundaries_path, f"{session_name}_Boundary_Points.npy"), boundary_points)

    coord_data = {
        "session": session_name, "X1": X1, "X2": X2, "Y1": Y1, "Y2": Y2,
        "width": X2 - X1, "height": Y2 - Y1,
        "boundary_points": boundary_points.tolist(),
        "derived_from_boundary": False,
        "synthesized_from_known_crop_bounds": True,
    }
    np.save(os.path.join(dlc_cropping_bounds_path, f"{session_name}_DLC_Cropping_Bounds.npy"), coord_data)

print("Done.")

Now that boundary/cropping files exist (synthesized above) and DLC output exists in `dlc_results\` under the matching `SessionXXXX` names, `batch_create_grids()` works unmodified.

In [6]:
now = datetime.now()
NUM_SQUARES = 10

print(f"\nCreating grids for {len(mouseinfo)} sessions (num_squares={NUM_SQUARES})...")

grid_results = dlc_utils.batch_create_grids(
    mouseinfo_df=mouseinfo,
    boundaries_directory=grid_boundaries_path,
    grid_files_directory=grid_path,
    cropping_directory=dlc_cropping_bounds_path,
    num_squares=NUM_SQUARES,
)
print(grid_results)


Creating grids for 21 sessions (num_squares=10)...
Batch grid creation started: 2026-07-27 13:29:06.923493
Grid size: 10 x 10
Creating grids for 21 sessions...
Boundary points directory: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\grid_boundaries
Cropping coordinates directory: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\dlc_cropping_bounds
Grid files directory: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\grid_files
Grid coordinates will be adjusted to cropped frame coordinate system

GRID CREATION SUMMARY
Total sessions processed: 21
Grids created: 0
Already existed: 21
Failed creation: 0
No boundary points: 0
No cropping coordinates: 0
Duration: 0:00:00.003085
{'total_sessions': 21, 'grids_created': 0, 'already_exists': 21, 'failed_creation': 0, 'no_boundaries': 0, 'no_cropping': 0, 'c

# Initial Visualizations

### Plot the Scatterplot with Grid overlayed for each trial

In [7]:
now = datetime.now()

print("\n--- Batch Processing Example ---")
batch_results = dlc_utils.batch_create_grid_scatter_plots(
    mouseinfo_df=mouseinfo,
    dlc_results_directory=dlc_results_path,
    grid_files_directory=grid_path,
    figures_directory=figure_path,
    dlc_scorer=dlc_scorer,
    bodypart='Belly',
    likelihood_threshold=0.6, # rough threshold to visualize when mice are in maze
    figure_size=(3, 3),
    show_plots=False,  # Don't display plots during batch processing
    rotate_180=True,  # matches the same rotation applied for grid_number assignment
)


--- Batch Processing Example ---
Batch creating grid scatter plots started: 2026-07-27 13:29:11.085128
Creating scatter plots for 21 sessions...
Bodypart: Belly
Likelihood threshold: 0.6
Saving plots to: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\figures\scatter_plots
-----------------------------
  Loaded DLC data: 234065 frames
  Plot saved to: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\figures\scatter_plots\Session0001_Belly_scatter_plot.png
✓ Plot created for Session0001
-----------------------------
  Loaded DLC data: 234065 frames
  Plot saved to: D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis\figures\scatter_plots\Session0002_Belly_scatter_plot.png
✓ Plot created for Session0002
-----------------------------
  Loaded DLC data: 234065 frames
  Plot saved to: D:\Gladstone Dropbox\Pal

### Create Trajectory Plots with Grid Overlaid

In [ ]:
now = datetime.now()

print("\n--- Different Colormaps Example ---")
colormaps = ['viridis']

for colormap in colormaps:
    print(f"\nCreating trajectory plots with {colormap} colormap...")
    batch_results = dlc_utils.batch_create_trajectory_plots(
        mouseinfo_df=mouseinfo,
        dlc_results_directory=dlc_results_path,
        grid_files_directory=grid_path,
        figures_directory=figure_path,  # batch_create_trajectory_plots() appends "trajectory_plots" itself -- passing figure_path + "trajectory_plots" here double-nested it
        dlc_scorer=dlc_scorer,
        bodypart='Belly',
        likelihood_threshold=0.6,
        colormap=colormap,
        show_plots=False,
        rotate_180=True,  # matches the same rotation applied for grid_number assignment
    )

# Create CSVs with Grid Numbers

`batch_append_grid_numbers()` also now works unmodified, as long as `bodyparts=["Belly"]` is passed explicitly (this maze's canonical tracked point — the default list uses lowercase names that won't match this data's capitalized DLC bodypart columns).

In [9]:
now = datetime.now()

# rotate_180=True: this camera's video is rotated 180 degrees relative to
# the maze orientation assumed in constants.py (start=top-left). Rather than
# rotating the videos or rebuilding the topology, we rotate the tracked
# coordinates into alignment before grid assignment -- grids/crop bounds/raw
# DLC tracking are unaffected and don't need to be redone.
grid_number_results = dlc_utils.batch_append_grid_numbers(
    mouseinfo_df=mouseinfo,
    dlc_results_directory=dlc_results_path,
    grid_files_directory=grid_path,
    dlc_scorer=dlc_scorer,
    bodyparts=["Belly"],
    save_directory=dlc_results_path,
    rotate_180=True,
)
print(grid_number_results)

Batch grid annotation started: 2026-07-27 13:29:53.449053

GRID ANNOTATION SUMMARY
Total sessions processed: 21
Successfully annotated: 0
Failed annotation: 0
Duration: 0:00:00.004001
{'total_sessions': 21, 'successfully_annotated': 0, 'failed_annotation': 0, 'successful_sessions': [], 'failed_sessions': []}
